In [1]:
with open('example1.smi') as f:
    data = [line.strip('\n') for line in f]
print('Number of generated molecules: ', len(data))

Number of generated molecules:  250


In [2]:
from rdkit import Chem
uniq = []
for smi in data:
    uniq_smi = Chem.MolToSmiles(Chem.MolFromSmiles(smi)) # standardize
    if uniq_smi not in uniq:
        uniq.append(uniq_smi)
print('Uniqueness: ', len(uniq)/len(data))

Uniqueness:  0.104


In [3]:
with open('example2_reinvent.csv') as f:
    all = [line.strip('\n').split(',')[10:15] for line in f if line.strip('\n').split(',')[0]][1:252]
qed = sum([float(p[0]) for p in all])/len([p[0] for p in all])
sa = sum([float(p[1]) for p in all])/len([p[1] for p in all])
sa_n = (10-sa)/10
slogp = sum([float(p[2]) for p in all])/len([p[2] for p in all])
stereocenters = sum([float(p[3]) for p in all])/len([p[3] for p in all])
similarity = sum([float(p[4]) for p in all])/len([p[4] for p in all])
print(f'QED = {qed:.3f}, SA = {sa:.3f}, slogp = {slogp:.3f}, stereo centers = {stereocenters:.3f}, similarity = {similarity:.3f}')
print(f'SA_normalized = {sa_n:.3f}')

QED = 0.118, SA = 4.873, slogp = 6.295, stereo centers = 3.956, similarity = 0.785
SA_normalized = 0.513


In [4]:
with open('example3_retro.csv') as f:
    retro = [int(line.strip('\n').split(',')[1]) for line in f if line.strip('\n').split(',')[0]]
print('Retro*: ', sum(retro)/len(retro))

Retro*:  0.888


In [5]:
with open('example4_glide.csv') as f:
    # assert the used column name == 'r_i_docking_score'
    glide = [line.strip('\n').split(',')[4] for line in f if line.strip('\n').split(',')[0]][1:]

glide_score = sum([float(g) for g in glide])/len(glide)
glide_n = (-5-glide_score)/10

# Here, only one score is retained for each molecule. 
# If you have multiple scores, make sure to remove duplicates and keep only the best score.
print('Glide_normalized: ', glide_n) 

Glide_normalized:  0.7589950879999968


In [6]:
suppl = Chem.SDMolSupplier('example5_pharm.sdf')
pharm = []
for mol in suppl:
    if mol:
        name = mol.GetProp ('_Name')
        pharm.append(float(mol.GetProp ('r_phase_PhaseScreenScore')))

pharm_value_n = (sum(pharm)/len(pharm))/2
# Here, only one score is retained for each molecule. 
# If you have multiple scores, make sure to remove duplicates and keep only the best score.
print(f'Pharm Num = {len(pharm)/250:.3f}, Pharm Value normalized = {pharm_value_n:.3f}')

Pharm Num = 0.604, Pharm Value normalized = 0.667


In [7]:
with open('example6_vina.csv') as f:
    vina = [float(line.strip('\n').split(',')[1]) for line in f if line.strip('\n').split(',')[0]]

vina_n = (-5-sum(vina)/len(vina))/7
print('Vina normalized: ', vina_n)

Vina normalized:  0.9754285714285723


In [8]:
print('Total score = QED + SA + similarity + Retro* + 0.5 * (dock_glide + dock_vina) + 0.5 * (pharm_num + pharm_value)')
print(f'= {qed + sa_n + similarity + sum(retro)/len(retro) + 0.5 * (glide_n + vina_n) + 0.5 * (len(pharm)/250+pharm_value_n):.3f}')

Total score = QED + SA + similarity + Retro* + 0.5 * (dock_glide + dock_vina) + 0.5 * (pharm_num + pharm_value)
= 3.806
